# Week3_ex3_to_4_analysis - Flux, EMF and Induced Current in a Ring Next to a Straight Wire

![Week3_ex3 result](Images/Week3_ex3_result_image.png)

A 400 mm copper wire carries 100 A at 10 Hz, and its ends touch the region's x faces, so it behaves like an infinite wire. One closed copper ring sits in the same plane at d = 200 mm: 36-sided, 9.5-10.5 mm circumradius, 1 mm x 1 mm section. Three quantities are pulled out. Flux is B·n integrated over a 10 mm disk, emf is E·t integrated around the 10 mm circle, and the ring current is J·n integrated over the ring's cross-section. (`coil_diameter = 10mm` in the original notebook is actually used as the radius.) Week3_ex4 is the same model with a 24-sided ring and PercentRefinement 30 instead of 15, and it exports the ring current too.

This is a light check. Nothing unexpected shows up, so there are no hypothesis steps. It covers three things: flux against the infinite-wire closed form, emf against Faraday's law inside the same FEM solution, and the small Phase 0 part of the ring current against the ring's own R-L time constant.

$$\Phi = \mu_0 I\left(d-\sqrt{d^2-a^2}\right),\qquad \mathrm{emf} = -j\omega\,\Phi,\qquad I_{ring} = \frac{\mathrm{emf}}{R + j\omega L}$$

## Week3_ex3 - flux and emf vs the closed form

![Week3_ex3 flux and emf vs phase](Images/Week3_ex3_flux_emf_vs_phase.png)

The original notebook's phase sweep is just Re(X·e^{jφ}) of one complex value per quantity. So every curve is a pure sinusoid and carries exactly two numbers, its Phase 0deg and 90deg values. The plot shows the main relation at a glance: the emf peaks 90deg after the flux, i.e. it lags by 90deg, as emf = -jωΦ says. The ring's own reaction is a few tenths of a percent, far too small to see here, so it's handled in tables below.

Flux and emf come from the CSV that the original notebook exported (Phase 0-360deg in 5deg steps). Only two phases are needed. At 0deg the flux peaks and the emf is near zero, and at 90deg it's the other way round. The disk area and the line length were read once from the Fields Calculator (`Integrate(Surface(field_circle), 1)` and `Integrate(Line(field_circle_line), 1)`). The line is a true circle to within 2 ppm, but the disk is a regular 24-gon with 1.14% less area. So Faraday's law inside the solution is checked both raw and with the disk area corrected.

The ex3 CSV has no `ring_current`, so it was read once from a Data Table in the AEDT GUI. At Phase 90deg it's -1.826 mA. The Phase 0deg value (0.000004 A) has only one significant digit at the table's 6 decimal places, so it isn't used. Inside the copper E = J/σ, so emf / I is the ring's resistance along the centerline and can be checked against the geometry directly.

In [1]:
import numpy as np
import pandas as pd

# 1. Model constants, closed-form flux, and the ring's R and L from geometry
mu0 = 4 * np.pi * 1e-7
sigma_cu = 5.8e7                  # S/m, AEDT copper
omega = 2 * np.pi * 10
I_wire = 100.0
d = 200e-3                        # ring center to wire
a = 10e-3                         # disk / line radius = ring centerline
t = 1e-3                          # ring section, 1 mm x 1 mm

phi_closed = mu0 * I_wire * (d - np.sqrt(d**2 - a**2))
area_24gon = 12 * a**2 * np.sin(np.pi / 12)

def ring_R(n_sides):
    path = n_sides * 2 * a * np.sin(np.pi / n_sides)
    width = t * np.cos(np.pi / n_sides)
    return path / (sigma_cu * width * t)

L_ring = mu0 * a * (np.log(8 * a / (0.447 * t)) - 2)    # thin-ring formula, square section as GMD

def at_phase(df, deg):
    return df[df["Phase [deg]"] == deg].iloc[0]

def check_table(df, area, n_sides, I_ring_90):
    p0, p90 = at_phase(df, 0), at_phase(df, 90)
    phi = abs(p0["ring_B []"])
    emf = abs(p90["ring_emf []"])
    area_ratio = area / (np.pi * a**2)
    R_eff = emf / abs(I_ring_90)
    checks = [
        ("disk area [m^2] vs true circle", area, np.pi * a**2),
        ("disk area [m^2] vs regular 24-gon", area, area_24gon),
        ("flux, Phase 0deg [Wb] vs closed form", phi, phi_closed),
        ("flux [Wb] vs closed form x meshed area", phi, phi_closed * area_ratio),
        ("emf, Phase 90deg [V] vs omega x closed-form flux", emf, omega * phi_closed),
        ("emf [V] vs omega x FEM flux (Faraday, raw)", emf, omega * phi),
        ("emf [V] vs omega x FEM flux / meshed area (Faraday)", emf, omega * phi / area_ratio),
        ("R_eff = emf / I [Ohm] vs geometry", R_eff, ring_R(n_sides)),
    ]
    return pd.DataFrame({
        "check": [c[0] for c in checks],
        "FEM": [c[1] for c in checks],
        "reference": [c[2] for c in checks],
        "difference [%]": [(c[1] / c[2] - 1) * 100 for c in checks],
    })

In [2]:
# 2. Fields Calculator values, read once in the AEDT GUI: Integrate(Surface(field_circle), 1), Integrate(Line(field_circle_line), 1)
area_3 = 0.000310582854123023
line_3 = 0.0628317495105722
print(f"line length vs 2*pi*a: {(line_3 / (2 * np.pi * a) - 1) * 1e6:+.1f} ppm")

# 3. ring_current at Phase 90deg, read once from a Data Table in the AEDT GUI (Setup1 : LastAdaptive)
I_ring_90_3 = -0.001826

ex3 = pd.read_csv("Week3_ex3_result.csv")
check_table(ex3, area_3, 36, I_ring_90_3)

line length vs 2*pi*a: -1.6 ppm


,check,FEM,reference,difference [%]
0,disk area [m^2] vs true circle,3.105829e-04,3.141593e-04,-1.138407e+00
1,disk area [m^2] vs regular 24-gon,3.105829e-04,3.105829e-04,-6.106227e-13
2,"flux, Phase 0deg [Wb] vs closed form",3.133708e-08,3.143559e-08,-3.133711e-01
3,flux [Wb] vs closed form x meshed area,3.133708e-08,3.107772e-08,8.345363e-01
4,"emf, Phase 90deg [V] vs omega x closed-form flux",1.996550e-06,1.975156e-06,1.083171e+00
5,"emf [V] vs omega x FEM flux (Faraday, raw)",1.996550e-06,1.968967e-06,1.400933e+00
6,emf [V] vs omega x FEM flux / meshed area (Far...,1.996550e-06,1.991640e-06,2.465773e-01
7,R_eff = emf / I [Ohm] vs geometry,1.093401e-03,1.086066e-03,6.753691e-01


## Week3_ex4 - 24-sided ring, ring current in the CSV

![Week3_ex4 result](Images/Week3_ex4_result_image.png)

The wire and the ring's position and size are unchanged. The ring is now 24-sided, and the setup uses PercentRefinement 30. The disk area was read the same way and is identical to ex3 (a regular 24-gon again). This time `ring_current` is in the exported CSV with full precision, so the Phase 0deg part of the current can be read directly. That's the part that lags because of the ring's own inductance.

![Week3_ex4 flux, emf and ring current vs phase](../Week3_ex4/Images/Week3_ex4_flux_emf_current_vs_phase.png)

The ring current peaks together with the emf, which is what you'd expect for a ring that's almost purely resistive. The lag from the ring's inductance is ωL/R ≈ 0.23%, about 0.13deg. That's invisible at this scale, so the table after the next cell reads it from the numbers instead.

In [3]:
# 4. Disk area, read once from the Fields Calculator in the AEDT GUI: Integrate(Surface(field_circle), 1)
area_4 = 0.000310582854123023

ex4 = pd.read_csv("../Week3_ex4/Week3_ex4_result.csv")
check_table(ex4, area_4, 24, at_phase(ex4, 90)["ring_current []"])

,check,FEM,reference,difference [%]
0,disk area [m^2] vs true circle,3.105829e-04,3.141593e-04,-1.138407e+00
1,disk area [m^2] vs regular 24-gon,3.105829e-04,3.105829e-04,-6.106227e-13
2,"flux, Phase 0deg [Wb] vs closed form",3.141556e-08,3.143559e-08,-6.370572e-02
3,flux [Wb] vs closed form x meshed area,3.141556e-08,3.107772e-08,1.087077e+00
4,"emf, Phase 90deg [V] vs omega x closed-form flux",2.009245e-06,1.975156e-06,1.725877e+00
5,"emf [V] vs omega x FEM flux (Faraday, raw)",2.009245e-06,1.973898e-06,1.790723e+00
6,emf [V] vs omega x FEM flux / meshed area (Far...,2.009245e-06,1.996628e-06,6.319304e-01
7,R_eff = emf / I [Ohm] vs geometry,1.104298e-03,1.089538e-03,1.354690e+00


## The ring's own reaction, read three ways

The same ωL/R shows up in three places. First, it's the Phase 0 part of the current (I lags the emf by ωL/R). Second, it's the Phase 0 part of the centerline emf. Third, it's the Phase 90 part of the disk flux. The table turns each ratio into an implied inductance using that exercise's R_eff = emf / I and compares it against the thin-ring formula. The sign conventions are set so that each route gives a positive number (value at phase φ = Re(X·e^{jφ})).

In [4]:
# 5. wL/R by route, for both rings
def routes(df, n_sides, I_ring_90, I_ring_0=None):
    p0, p90 = at_phase(df, 0), at_phase(df, 90)
    R_eff = abs(p90["ring_emf []"] / I_ring_90)
    ratios = {"formula (geometry R, thin-ring L)": omega * L_ring / ring_R(n_sides)}
    if I_ring_0 is not None:
        ratios["current: -I(0deg) / I(90deg)"] = -I_ring_0 / I_ring_90
    ratios["centerline emf: -emf(0deg) / emf(90deg)"] = -p0["ring_emf []"] / p90["ring_emf []"]
    ratios["disk flux: flux(90deg) / flux(0deg)"] = p90["ring_B []"] / p0["ring_B []"]
    rows = []
    for name, r in ratios.items():
        L_implied = L_ring if name.startswith("formula") else r * R_eff / omega
        rows.append({"route": name, "wL/R [%]": r * 100, "implied L [nH]": L_implied * 1e9,
                     "vs formula L [%]": (L_implied / L_ring - 1) * 100})
    return pd.DataFrame(rows)

p0_4, p90_4 = at_phase(ex4, 0), at_phase(ex4, 90)
pd.concat({
    "Week3_ex3 (36-sided)": routes(ex3, 36, I_ring_90_3),
    "Week3_ex4 (24-sided)": routes(ex4, 24, p90_4["ring_current []"], p0_4["ring_current []"]),
})

route  wL/R [%]  \
Week3_ex3 (36-sided) 0        formula (geometry R, thin-ring L)  0.231711   
                     1  centerline emf: -emf(0deg) / emf(90deg)  0.245410   
                     2      disk flux: flux(90deg) / flux(0deg)  0.269823   
Week3_ex4 (24-sided) 0        formula (geometry R, thin-ring L)  0.230972   
                     1             current: -I(0deg) / I(90deg)  0.233595   
                     2  centerline emf: -emf(0deg) / emf(90deg)  0.238986   
                     3      disk flux: flux(90deg) / flux(0deg)  0.261817   

                        implied L [nH]  vs formula L [%]  
Week3_ex3 (36-sided) 0       40.051829          0.000000  
                     1       42.706238          6.627434  
                     2       46.954606         17.234609  
Week3_ex4 (24-sided) 0       40.051829          0.000000  
                     1       41.055412          2.505710  
                     2       42.002905          4.871378  
                     3       46.015576         14.890073

## Conclusion

**Week3_ex3.** Flux through the 10 mm disk comes out 0.31% below the infinite-wire closed form. That small number hides two larger pieces pulling in opposite directions. The meshed disk is a regular 24-gon with 1.14% less area, and once that's taken out, the field itself sits 0.83% above the closed form. The emf is measured on a line that is a true circle to within 2 ppm, and it sits 1.08% above. Faraday's law inside the same solution closes to +0.25% once the disk area is accounted for. Neither the field offset nor the leftover is tested here. One candidate for the offset is that this model has no refinement box in the air around the ring (only the ring and the wire get length-based mesh operations), so the air mesh there is set by the adaptive passes alone. R_eff = emf / I is 1.093 mΩ, within 0.7% of the 36-sided ring's geometric resistance.

**Week3_ex4.** The 24-sided ring gives the same picture. Flux is -0.06% raw and +1.10% after the area correction, emf is +1.72%, and the Faraday leftover is +0.63%. The shift from ex3 comes with two changes at once (ring polygon and PercentRefinement), so it isn't read as a trend. What ex4 adds is the ring current at full precision. Its Phase 0deg part is 0.234% of the Phase 90deg part, which is ωL/R of the ring. With R_eff = 1.104 mΩ, that gives L ≈ 41 nH, against 40 nH from the thin-ring formula. The centerline-emf route gives 42-43 nH, so the ring behaves like the textbook R-L loop. The disk-flux route reads 15-17% high in both exercises. By Faraday's law, the centerline emf and the flux through the disk on that same circle should give the same number. So this gap is the same surface-vs-line mismatch seen in the Phase 0 part, not a problem with the ring model. It's much larger in relative terms here, and one candidate (not tested) is that the ring's own field is concentrated right at the conductor, which is exactly where the disk edge runs.

Neither exercise needs a closer look. The flux and emf match the closed form at the 1% level once the meshed disk area is accounted for, and the induced current matches the ring's R and L computed from geometry.